CLEANING RAW DATA AND CONVERTING IT TO APPROPRIATE DATA TYPES

In [18]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
import os
from datetime import datetime

In [23]:
spark = SparkSession.builder \
    .appName("raw_train_live_status") \
    .config("spark.jars.packages","io.delta:delta-spark_2.13:4.0.0") \
    .config("spark.sql.extensions","io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog","org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.databricks.delta.properties.defaults.enableDeletionVectors", "true") \
    .master("local[*]") \
    .getOrCreate()


current_date = datetime.now().date()

In [ ]:
df = spark.read.format("delta")\
    .load(os.path.join(os.path.abspath(""),f"raw_data\\raw_batch_data\\{current_date}\\trainscheduledtime"))

In [33]:
cleaned_df = df.withColumn("trainNumber",col("trainNumber").cast(StringType()))\
    .withColumn("createdDate",col("createdDate").cast(TimestampType()))

In [35]:
cleaned_df.write \
    .format("delta") \
    .mode("append") \
    .partitionBy("createdDate") \
    .option("mergeSchema",True)\
    .save(os.path.join(os.path.abspath(""),f"silver_data\\batch_data\\trainscheduledtime_cleaned"))